# `/generate` Pipeline Walkthrough

Interactive, cell-by-cell prototype of what `POST /generate` will do. Unlike `notebooks/upload_pipeline_walkthrough.ipynb`, **this notebook makes real calls**: it reads the sender/receiver documents you already uploaded (Table Storage) and calls Azure OpenAI to generate a structured article.

Everything here is defined inline, not imported from `app/`. That's deliberate — this is where the `/generate` design gets prototyped and tuned (prompt wording, retry behavior, image-slot logic) before it gets lifted into `app/models.py`, `app/generation.py`, and `app/routers/generate.py` as its own pass, the same way `app/routers/upload.py` already exists. Once you're happy with how this runs end to end, say so and that extraction happens next.

**Prerequisites before you run this top to bottom:**
1. `POST /upload` already run for at least one sender + one receiver (the walkthrough below defaults to `northbridge-analytics` / `ferrow-industrial`, per `azure-setup-log.md` §3 — already tested ✅).
2. An Azure OpenAI resource provisioned with a `gpt-5-mini` deployment, and `.env` filled in with its endpoint + key. **Not done yet** as of `azure-setup-log.md` §4 — Section 0 below walks through it.
3. `pip install openai` in your `.venv` (added to `requirements.txt`).

Run cells top to bottom, in order.

## Section 0 — Provision the Azure OpenAI resource

This is the one piece of Azure infrastructure this endpoint needs, if it isn't already provisioned. Do this once, before running the generation cells further down — everything up through Section 2 (context fetch) works without it if you want to run those first.

**Why Azure OpenAI and not a personal OpenAI key**: consistent with the rest of the architecture (Blob + Table Storage already on Azure) rather than mixing in a separate vendor for one piece.

### Steps (Azure Portal)

1. **Create the resource.** Portal → "Create a resource" → search **"Azure OpenAI"** → Create.
   - **Subscription**: your personal one (same $200/30-day credit from `azure-setup-log.md` §1).
   - **Resource group**: the resource group — the same one Storage is already in. Keeps everything under one deletable container.
   - **Region**: `East US` — same region as `the storage account`, and confirmed to support `gpt-5-mini` deployments. Keeping Storage and OpenAI in the same region also avoids a cross-region latency/egress footnote you'd otherwise have to explain live.
   - **Name**: e.g. `the Azure OpenAI resource` — becomes part of the endpoint URL.
   - **Pricing tier**: `Standard S0` (the only tier offered — you pay per token, not a flat fee, so this is still cost-conscious for low weekend query volume).
   - Review + Create.

2. **Deploy the model.** Once the resource exists, open it → **"Explore Foundry portal"**. If that drops you into the multi-resource "All resources" list instead of a view scoped to your resource, click the `the Azure OpenAI resource` row → **"Open in Foundry Classic"** (formerly "Azure OpenAI Studio" — same destination, Microsoft renamed the portal). Left nav → **Deployments** → **+ Deploy model** → **Deploy base model**.
   - **Model**: search "mini" and pick a **currently supported** small/cheap chat-completion model — confirm there's no deprecation warning before proceeding. `gpt-4o-mini` (the original plan here) turned out to be retired from Azure as of March 2026 — see `azure-setup-log.md` §4's "Deployment attempt 1" note. `gpt-5-mini` is the current pick.
   - **Deployment type**: **`Standard`**, not `Global Standard`. This matters more than it looks: with `Global Standard` selected, the deploy dialog's "AI resource" field can silently read `(create) <auto-name>-eastus2` — meaning Azure is about to spin up a *second*, differently-named Cognitive Services resource in a different region instead of using `the Azure OpenAI resource`. Check that field before clicking deploy; `Standard` is what keeps this inside the one resource this file already tracks.
   - **Deployment name**: use exactly the model name (e.g. `gpt-5-mini`) — matches `AZURE_OPENAI_DEPLOYMENT_NAME` in `.env.example`, so you don't need to change that default. (Azure OpenAI's API calls use this *deployment name*, not the base model name — easy gotcha, called out again in Section 4 where the code actually uses it.)
   - **Tokens-per-minute quota**: click **Customize** and set a real nonzero value (e.g. `10` = 10K TPM) — the dialog can default to showing `0 tokens per minute`, which deploys "successfully" but is unusable.

3. **Copy the credentials.** Back on the resource → left nav → **"Keys and Endpoint"**:
   - `AZURE_OPENAI_ENDPOINT` = the **Endpoint** value (looks like `https://the Azure OpenAI resource.openai.azure.com/`).
   - `AZURE_OPENAI_API_KEY` = **KEY 1**.
   - Paste both into `.env` (already has placeholders). Set `AZURE_OPENAI_DEPLOYMENT_NAME=gpt-5-mini` and `AZURE_OPENAI_API_VERSION=2024-12-01-preview` — Azure's own "Get Started" code sample on the deployment's Details page (Foundry Classic → Deployments → gpt-5-mini) generates that exact api_version, so it's taken as authoritative for this specific model/resource rather than guessed.

4. **Restart the Jupyter kernel** after editing `.env` so `Settings()` picks up the new values (it only reads `.env` once, at import time).

5. Once confirmed working (Section 4 below returns a real completion), update `azure-setup-log.md` §4 the same way §3 (Storage) is documented — resource name, region, status — so the provisioning record stays complete for the presentation.

## Setup
Same pattern as `upload_pipeline_walkthrough.ipynb`: add the repo root to `sys.path` so `from app...` imports resolve, then confirm `openai` is installed.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent
sys.path.insert(0, str(REPO_ROOT))

print("Notebook cwd:", Path.cwd())
print("Repo root added to sys.path:", REPO_ROOT)
# If imports below fail with ModuleNotFoundError: openai, run in a terminal (with .venv active):
#   pip install openai
# then restart this kernel.

In [ ]:
from app.config import settings

def _mask(value: str, keep: int = 4) -> str:
    if not value:
        return "(empty)"
    return value[:keep] + "..." + value[-keep:] if len(value) > keep * 2 else "***"

print("AZURE_STORAGE_CONNECTION_STRING set:", bool(settings.azure_storage_connection_string))
print("AZURE_OPENAI_ENDPOINT:", settings.azure_openai_endpoint or "(empty)")
print("AZURE_OPENAI_API_KEY:", _mask(settings.azure_openai_api_key))
print("AZURE_OPENAI_DEPLOYMENT_NAME:", settings.azure_openai_deployment_name)
print("AZURE_OPENAI_API_VERSION:", settings.azure_openai_api_version)

# Sections 0-2 (provisioning + context fetch) only need AZURE_STORAGE_CONNECTION_STRING.
# Section 4 onward (the actual LLM call) needs AZURE_OPENAI_ENDPOINT + AZURE_OPENAI_API_KEY too —
# if those are still empty, finish Section 0 above before running past Section 3.

## Section 1 — The contract: `GenerateRequest` and `ArticleOutput`

Defined here field-by-field, matching the layout/schema contract in `docs/architecture/api-payload-schemas.drawio` exactly — this is just the Pydantic implementation of it. Every field is tagged in a comment with its category:

- **TEMPLATE-FIXED** — locked by the layout template, identical for every pair.
- **INPUT PARAMETER** — supplied by the caller on the request, echoed into the response.
- **LLM-GENERATED** — written by the model for this specific pair.
- **SYSTEM-ASSIGNED** — assigned by this backend code, not authored by the LLM or the caller.

Splitting `ArticleOutput` (the full response) from `LLMDraft` (just the LLM-authored subset, Section 4) is the mechanism that keeps the model from being trusted with IDs, timestamps, or the caller's own theme colors — the LLM only ever sees a request to produce `LLMDraft`; everything else is filled in by this code afterward.

## The Pydantic model pipeline, at a glance

Six Pydantic models get defined below. Before going through each one, here's where every one of them sits in the request → generation → response flow, and *why* there are six instead of one big class:

```
HTTP client (Postman / a frontend / external evaluators in Swagger)
        │  sends raw, untyped JSON
        ▼
┌───────────────────────────────────────────────────────────────┐
│ GenerateRequest                    ← validated INCOMING boundary        │
│   sender_id, receiver_id : str                                       │
│   theme : ThemeColors              ← nested model, hex checked          │
│   feedback : str | None                                              │
└───────────────────────────────────────────────────────────────┘
        │  sender_id + receiver_id → look up stored context (Section 2)
        ▼
  sender_ctx, receiver_ctx        (plain dicts — not Pydantic; raw text/
                                    tables/image paths straight from Table
                                    Storage, nothing to validate yet)
        │  fed into the prompt (Section 4) → Azure OpenAI call (Section 5)
        ▼
┌──────────────────────────────────────────────────────────────┐
│ LLMDraft                    ← validated MODEL-OUTPUT boundary           │
│   headline, subheadline, call_to_action : str  (word limits)         │
│   body_sections : list[BodySection]            ← nested, reused        │
│   *_logo_caption, contextual_image_caption : str                     │
└───────────────────────────────────────────────────────────────┘
        │  Section 6: validation failure here → error text fed back to
        │  the model, retried (this is the ONLY model the LLM ever
        │  produces — it never sees article_id, version, or theme)
        │  Section 7: backend fills in everything else
        ▼
┌──────────────────────────────────────────────────────────────┐
│ ArticleOutput               ← validated FINAL RESPONSE                  │
│   article_id, version, created_at      SYSTEM-ASSIGNED (backend)     │
│   sender_id, receiver_id, theme        INPUT PARAMETER (copied      │
│                                         straight from GenerateRequest)│
│   headline, subheadline,               LLM-GENERATED (copied        │
│   body_sections, call_to_action        straight from LLMDraft)      │
│   image_slots : list[ImageSlot]        assembled in Section 3       │
│   total_word_count                     SYSTEM-ASSIGNED, computed    │
│   status                               TEMPLATE-FIXED, always "draft"│
└──────────────────────────────────────────────────────────────┘
```

**Shared building-block models** (not boundaries on their own — typed pieces reused *inside* the boundary models above):
- `ThemeColors` — appears inside `GenerateRequest.theme` (validated coming in) *and* `ArticleOutput.theme` (same values, just echoed back out).
- `BodySection` — appears inside `LLMDraft.body_sections` (validated coming out of the model) *and* `ArticleOutput.body_sections` (the exact same validated objects, copied over in Section 7 — not rebuilt or re-typed).
- `ImageSlot` — only ever appears inside `ArticleOutput.image_slots`. It doesn't exist on `LLMDraft` at all — it's assembled in Section 3, after the LLM call, from the LLM's captions plus blob paths this code resolves itself.

**The key thing to notice**: Pydantic validation isn't a one-time thing that happens "at the API." It happens *every single time* one of these classes gets constructed — `GenerateRequest(...)`, `LLMDraft(...)` (built by the openai SDK when it parses the model's response), `ArticleOutput(...)` (built by `assemble_article()`). Three separate checkpoints, not one, each catching a different kind of mistake as early as possible.

## How to read every model below: what `@field_validator` and `@classmethod` are doing

Every model below follows the same shape, so this is worth explaining once instead of six times:

```python
class ThemeColors(BaseModel):
    primary_color: str

    @field_validator("primary_color")
    @classmethod
    def _must_be_hex(cls, v: str) -> str:
        if not _HEX_COLOR_RE.match(v):
            raise ValueError(...)
        return v
```

- **`primary_color: str`** is a normal Pydantic field. On its own, Pydantic already checks the *type* — it'll reject `primary_color=123` (an int) — but it has no idea what a valid hex color looks like. That's a business rule, not a type.
- **`@field_validator("primary_color")`** registers `_must_be_hex` as extra, custom logic that runs *after* the type check, specifically for that field. You can name multiple fields in one call (`LLMDraft`'s validators below are reused across `headline`, `subheadline`, `call_to_action` — one decorator each, same pattern) or reuse one validator function for several fields at once.
- **`@classmethod`** is required by Pydantic v2's API for `@field_validator` — the function receives `cls` (the class itself, e.g. `ThemeColors`) rather than `self` (an instance), because it runs *while* the instance is still being built, before there's a fully-formed object to call a normal method on. You never call `_must_be_hex` yourself; Pydantic calls it internally, automatically, every single time someone writes `ThemeColors(primary_color="...")`.
- **The `raise ValueError(...)` / `return v` pattern**: return the value (optionally transformed) if it's valid; raise `ValueError` if it's not — Pydantic wraps that into its own `ValidationError`. This exact mechanism is what Section 6's retry loop catches.

Everything below — `ThemeColors`'s hex check, `LLMDraft`'s word-limit checks, `BodySection`'s 150-word cap — uses this identical pattern for a business rule that JSON's own type system can't express (a hex format, a word count, a count of list items).

In [ ]:
import json
import re
from datetime import datetime, timezone
from typing import Literal
from uuid import UUID, uuid4

from pydantic import BaseModel, Field, ValidationError, field_validator

## `word_count` and `ThemeColors`

`word_count` is a plain helper function, not a Pydantic model — `text.split()` on whitespace, then a count. It backs every word-limit validator that follows; it is **not** a token count (an LLM's internal tokenization is a different, finer-grained unit — the schema PDF's limits are phrased as "words," so this matches that literally).

`ThemeColors` is the header/CTA/accent-band colors — **INPUT PARAMETER**, not LLM output. The caller (whoever calls `POST /generate`) supplies these; the LLM never sees or writes them, and this code never invents them — they're only ever copied straight through from `GenerateRequest.theme` into `ArticleOutput.theme` (see the workflow diagram above). The validator's job is narrow: catch a malformed hex string (e.g. a client sending `"blue"`, or `"1B3A5C"` without the `#`) at the moment `ThemeColors(...)` gets constructed, with a clear error — instead of that bad value silently flowing all the way through to the rendered article and breaking the CTA band's color at render time, several steps and one API round-trip later.

In [ ]:
def word_count(text: str) -> int:
    """Simple whitespace-split word count — matches how the layout template's word limits
    (“≤ 12 words” etc.) are meant to be read; not a token count."""
    return len(text.split())


_HEX_COLOR_RE = re.compile(r"^#[0-9A-Fa-f]{6}$")


class ThemeColors(BaseModel):
    """INPUT PARAMETER — supplied by the caller on /generate, echoed back into
    ArticleOutput.theme. Drives header/CTA/accent bands in the rendered layout
    (see docs/architecture/api-payload-schemas.drawio)."""

    primary_color: str
    secondary_color: str
    accent_color: str

    @field_validator("primary_color", "secondary_color", "accent_color")
    @classmethod
    def _must_be_hex(cls, v: str) -> str:
        if not _HEX_COLOR_RE.match(v):
            raise ValueError(f"'{v}' is not a 6-digit hex color like #1B3A5C")
        return v

## `GenerateRequest` — and why it needs validation at all

It can look redundant to Pydantic-validate `sender_id`, `receiver_id`, and `theme` here — in this notebook, *we* write the exact values a few cells down (Section 7), so of course they're valid; we picked them ourselves.

That's true *in this notebook specifically*. `GenerateRequest`'s real job is different: it's the boundary for **`POST /generate`'s actual HTTP caller** — Postman, a frontend, or external evaluators poking around in Swagger — none of whom are this notebook. Raw HTTP JSON arrives completely untyped: a required field can be missing, `theme` could arrive as `{"primary_color": "not-a-color"}`, or `sender_id` could be sent as a number by a buggy client. `GenerateRequest` is what turns "someone sent some JSON" into "we are now guaranteed `sender_id` is a string, `theme.primary_color` is a valid hex code" — checked once, at the very first line of the handler, before any business logic (a Table Storage lookup, an Azure OpenAI call — both costing real time and money) ever runs.

Once this gets extracted into `app/routers/generate.py`, `GenerateRequest` becomes the actual `request: GenerateRequest` parameter in the endpoint's function signature — FastAPI then does this parsing/validation *automatically*, returns a clean `422` with a specific error message if the JSON doesn't match, and generates this endpoint's `/docs` (Swagger) schema from the model, for free. None of that exists yet with a plain dict.

`sender_id` / `receiver_id` are also the two fields that matter most functionally, independent of validation: they're the pair key `fetch_context()` (Section 2) looks up in Table Storage — everything downstream (which two document sets get pulled, what the LLM sees, what gets generated) depends on getting these two strings right.

In [ ]:
class GenerateRequest(BaseModel):
    """Request body of POST /generate."""

    sender_id: str  # INPUT PARAMETER — pairs with an existing /upload role="sender" batch
    receiver_id: str  # INPUT PARAMETER — pairs with an existing /upload role="receiver" batch
    theme: ThemeColors  # INPUT PARAMETER
    feedback: str | None = None  # INPUT PARAMETER, optional — from a prior /evaluate call
    # (decision-log.md §9: /generate never calls /evaluate itself; feedback is a plain string the
    # caller passes back in, same shape as /evaluate's response.feedback)

## `BodySection` and `ImageSlot` — the two reusable pieces

Neither of these is a "boundary" model on its own (nothing external ever sends or receives a bare `BodySection` or `ImageSlot`) — they're typed building blocks nested inside the boundary models above, so a list of them gets validated item-by-item instead of as one opaque blob.

- **`BodySection`** = one section of the article body (a heading + its text). `LLMDraft.body_sections` and `ArticleOutput.body_sections` are both `list[BodySection]` — the *same* validated objects, just copied across in Section 7, not rebuilt. Its own validator enforces the schema PDF's **≤150 words per section** rule (TEMPLATE-FIXED) at the moment each section gets constructed.
- **`ImageSlot`** = one of the article's 3 fixed image slots. `slot_id` and `source_type` are `Literal[...]` types — Python's way of saying "only these exact string values are allowed" — which is how the TEMPLATE-FIXED "always `sender_logo`, `receiver_logo`, `contextual`, in that order" rule from the schema PDF gets enforced structurally, not just described in a comment. `blob_path` is SYSTEM-ASSIGNED (this code resolves it from Table Storage, in Section 3); `caption` is LLM-GENERATED. Unlike `BodySection`, `ImageSlot` only ever appears on `ArticleOutput` — `LLMDraft` has no image-slot concept at all, only the three caption strings that later feed into building one.

In [ ]:
class BodySection(BaseModel):
    heading: str  # LLM-GENERATED
    text: str  # LLM-GENERATED

    @field_validator("text")
    @classmethod
    def _max_150_words(cls, v: str) -> str:
        n = word_count(v)
        if n > 150:
            raise ValueError(f"body_sections[].text must be <=150 words, got {n}")
        return v


ImageSlotId = Literal["logo_sender", "logo_receiver", "hero_contextual"]
ImageSourceType = Literal["sender_logo", "receiver_logo", "contextual"]


class ImageSlot(BaseModel):
    slot_id: ImageSlotId  # TEMPLATE-FIXED
    source_type: ImageSourceType  # TEMPLATE-FIXED
    blob_path: str | None  # SYSTEM-ASSIGNED — resolved from upload; None = not resolved (see Section 3)
    caption: str  # LLM-GENERATED

## `LLMDraft` — the one thing the model is allowed to author

Full reasoning is in the code's own docstring just below (worth reading — it explains the `Field(min_length=..., max_length=...)` vs `@field_validator` split specifically). Short version: this is the *only* Pydantic model that ever gets sent to Azure OpenAI as `response_format` (Section 5). Every field on it is LLM-GENERATED. It has no `article_id`, no `theme`, no `sender_id` — the model is structurally incapable of writing those, because it's never asked to.

In [ ]:
class LLMDraft(BaseModel):
    """The subset of ArticleOutput the model is responsible for authoring — every field here is
    LLM-GENERATED per the schema PDF's legend. This, not ArticleOutput, is what gets passed as
    `response_format` to Azure OpenAI in Section 4 — the model is never shown article_id, version,
    sender_id/receiver_id, or theme, so it cannot hallucinate or overwrite them.

    Two different enforcement mechanisms are deliberately mixed here:
    - `body_sections`'s 2-3 item count uses Field(min_length=2, max_length=3) — this becomes part of
      the JSON Schema sent to Azure OpenAI's structured-output mode, so the API itself guarantees the
      count structurally, before this code ever sees a response.
    - Word limits (“<=12 words”, “<=150 words”, ...) use @field_validator instead — JSON Schema
      has no “word count” primitive (only character length, which isn't the same thing), so these
      can only be checked in Python, after the API returns a structurally-valid response. This is
      exactly the gap Section 5's retry loop exists to cover — structurally valid JSON that still
      breaks a business rule the schema itself couldn't express.
    """

    headline: str  # LLM-GENERATED, <=12 words
    subheadline: str | None = None  # LLM-GENERATED, optional, <=20 words
    body_sections: list[BodySection] = Field(min_length=2, max_length=3)  # TEMPLATE-FIXED count
    call_to_action: str  # LLM-GENERATED, <=25 words
    sender_logo_caption: str  # LLM-GENERATED — typically just the sender's display name
    receiver_logo_caption: str  # LLM-GENERATED — typically just the receiver's display name
    contextual_image_caption: str  # LLM-GENERATED — describes what the hero image should depict

    @field_validator("headline")
    @classmethod
    def _headline_max_12(cls, v: str) -> str:
        n = word_count(v)
        if n > 12:
            raise ValueError(f"headline must be <=12 words, got {n}: '{v}'")
        return v

    @field_validator("subheadline")
    @classmethod
    def _subheadline_max_20(cls, v: str | None) -> str | None:
        if v is None:
            return v
        n = word_count(v)
        if n > 20:
            raise ValueError(f"subheadline must be <=20 words, got {n}: '{v}'")
        return v

    @field_validator("call_to_action")
    @classmethod
    def _cta_max_25(cls, v: str) -> str:
        n = word_count(v)
        if n > 25:
            raise ValueError(f"call_to_action must be <=25 words, got {n}: '{v}'")
        return v

## `ArticleOutput` — the full response, assembled, not generated

This is what `POST /generate` actually returns — and notice nothing here gets typed by the LLM directly. Section 7's `assemble_article()` builds this by combining: `LLMDraft`'s fields (LLM-GENERATED), `GenerateRequest`'s fields (INPUT PARAMETER, echoed), and fresh values this code computes itself (SYSTEM-ASSIGNED — `article_id`, `version`, `total_word_count`, `created_at`). No single source produces this object end-to-end — that's deliberate, per the workflow diagram above.

In [ ]:
class ArticleOutput(BaseModel):
    """Response body of POST /generate — the full public contract. Field-for-field match of
    docs/architecture/api-payload-schemas.drawio."""

    article_id: UUID  # SYSTEM-ASSIGNED
    sender_id: str  # INPUT PARAMETER, echoed
    receiver_id: str  # INPUT PARAMETER, echoed
    version: int  # SYSTEM-ASSIGNED, auto-incremented per (sender_id, receiver_id) pair
    headline: str  # LLM-GENERATED
    subheadline: str | None = None  # LLM-GENERATED
    body_sections: list[BodySection]  # TEMPLATE-FIXED count (already enforced via LLMDraft)
    call_to_action: str  # LLM-GENERATED
    image_slots: list[ImageSlot] = Field(min_length=3, max_length=3)  # TEMPLATE-FIXED, always 3
    theme: ThemeColors  # INPUT PARAMETER, echoed
    total_word_count: int  # SYSTEM-ASSIGNED, computed — target 300-600 (soft, checked in Section 7)
    status: Literal["draft"] = "draft"  # TEMPLATE-FIXED — no publish workflow in this prototype
    created_at: datetime  # SYSTEM-ASSIGNED

In [ ]:
# Sanity-check the contract by printing the JSON Schema each model produces — this is literally
# what FastAPI will expose at /docs once these models back a real endpoint, and it's the same
# schema (for LLMDraft) that gets sent to Azure OpenAI in Section 4.
for name, model in [("GenerateRequest", GenerateRequest), ("LLMDraft", LLMDraft), ("ArticleOutput", ArticleOutput)]:
    print(f"--- {name} ---")
    print(json.dumps(model.model_json_schema(), indent=2)[:800])
    print("...\n")

## Section 2 — Fetching context: structured lookup, not RAG

Per `decision-log.md` §6: each `/generate` call has exactly two known, bounded document sets, identified by `sender_id` / `receiver_id` — nothing to semantically search for. "Retrieval" here means an exact lookup against Table Storage by the same `PartitionKey = f"{sender_id}__{receiver_id}"` scheme `storage.save_document_metadata()` already writes (see `CODE_MAP.md`), filtered further by `role`. A vector DB would be solving a problem this system doesn't have.

In [ ]:
from azure.data.tables import TableServiceClient

METADATA_TABLE = "documentmetadata"  # same table /upload already writes to


def _table_client(table_name: str):
    # Fresh client per call, same deliberate simplicity trade-off app/storage.py already makes
    # (see its module docstring) — no shared global client to worry about init-order or thread-safety for.
    service = TableServiceClient.from_connection_string(settings.azure_storage_connection_string)
    service.create_table_if_not_exists(table_name)  # idempotent — no-ops if it already exists
    return service.get_table_client(table_name)


def fetch_context(sender_id: str, receiver_id: str, role: str) -> dict:
    """Looks up every previously-uploaded document for one side of this pair and merges them into
    one context bundle. Raises a clear error (not an empty result) if nothing was uploaded — fail
    loud, same policy as parsing.py's PdfParseError (decision-log.md §6).

    Deduplicates by exact `text` match before merging. Uploading the same PDF twice under the same
    (sender_id, receiver_id, role) writes two separate rows to `documentmetadata` — each /upload
    call assigns a fresh document_id regardless of content, there's no upload-time uniqueness check
    (see `upload.py`'s docstring on sender_id/receiver_id being plain, unvalidated strings). Left
    unfiltered, that duplicate text would be sent to the LLM twice — wasted tokens, and a real risk
    of skewing the generated article toward whatever got accidentally repeated.

    Fixed here, at read time, rather than in /upload at write time: /upload is already built and
    Postman-tested end-to-end (azure-setup-log.md §3) — reopening it for a pre-write existence check
    is real surgery to a settled, working piece for a prototype under a hard deadline. This function
    is still being iterated on, so it's the cheaper, better-scoped place to fix the symptom that
    actually matters (duplicate content reaching the LLM). A write-time fix (reject/overwrite a
    duplicate upload before it's ever stored) would be the more complete production answer — noted,
    not built, same pattern as this notebook's other named-not-solved gaps.

    Never silent: a skipped duplicate is printed, not just dropped — same fail-loud policy as
    everything else here.
    """
    partition_key = f"{sender_id}__{receiver_id}"
    table = _table_client(METADATA_TABLE)
    entities = list(
        table.query_entities(query_filter=f"PartitionKey eq '{partition_key}' and role eq '{role}'")
    )
    if not entities:
        raise ValueError(
            f"No {role!r} documents found for sender_id={sender_id!r} receiver_id={receiver_id!r}. "
            f"Upload context PDFs via POST /upload first (see upload_pipeline_walkthrough.ipynb)."
        )

    texts, tables, image_paths, filenames = [], [], [], []
    seen_texts: set[str] = set()
    duplicates_skipped = 0
    for e in entities:
        text = e.get("text", "")
        if text in seen_texts:
            duplicates_skipped += 1
            print(
                f"fetch_context: skipping duplicate {role} document {e.get('filename')!r} "
                f"(document_id={e.get('RowKey')}) — identical text already included from an "
                f"earlier upload of this pair."
            )
            continue
        seen_texts.add(text)
        texts.append(text)
        tables.extend(json.loads(e.get("tables_json", "[]")))
        image_paths.extend(json.loads(e.get("image_blob_paths_json", "[]")))
        filenames.append(e.get("filename"))

    return {
        "role": role,
        "document_count": len(filenames),  # after dedup — unique documents actually merged
        "duplicates_skipped": duplicates_skipped,
        "filenames": filenames,
        "text": "\n\n".join(texts),
        "tables": tables,
        "image_paths": image_paths,
    }

In [ ]:
SENDER_ID = "northbridge-analytics"
RECEIVER_ID = "ferrow-industrial"

sender_ctx = fetch_context(SENDER_ID, RECEIVER_ID, role="sender")
receiver_ctx = fetch_context(SENDER_ID, RECEIVER_ID, role="receiver")

for name, ctx in [("Sender", sender_ctx), ("Receiver", receiver_ctx)]:
    print(f"{name}: {ctx['document_count']} doc(s) {ctx['filenames']}, "
          f"{len(ctx['text'])} chars text, {len(ctx['tables'])} table(s), {len(ctx['image_paths'])} image(s), "
          f"{ctx['duplicates_skipped']} duplicate(s) skipped")
    print("  image_paths:", ctx["image_paths"])

In [ ]:
sender_ctx

## Section 3 — Resolving image slots

Fixed 3-slot order, per the schema PDF: `logo_sender`, `logo_receiver`, then one `hero_contextual` slot. Logo paths are resolved deterministically — the first image extracted from each side's uploaded documents — no vision model involved, since the mock PDFs each embed exactly one image (their letterhead logo).

**Known gap, named rather than papered over** (extends `decision-log.md` §7's already-flagged image-captioning gap): there is no *contextual* image asset ingested anywhere in this pipeline — `/upload` only ever sees the sender/receiver's own letterhead PDFs, not a stock-photo or generated-image source. `blob_path` for `hero_contextual` is left `None` below rather than faked. A real build would need either a stock-image search step or a text-to-image call (e.g. DALL-E via the same Azure OpenAI resource) feeding this slot from the LLM's `contextual_image_caption`. Worth naming explicitly if asked — it's a real limitation of the "lightweight prototype" scope, not an oversight.

In [ ]:
def resolve_image_slots(sender_ctx: dict, receiver_ctx: dict, draft: "LLMDraft") -> list[ImageSlot]:
    sender_logo_path = sender_ctx["image_paths"][0] if sender_ctx["image_paths"] else None
    receiver_logo_path = receiver_ctx["image_paths"][0] if receiver_ctx["image_paths"] else None

    return [
        ImageSlot(slot_id="logo_sender", source_type="sender_logo",
                  blob_path=sender_logo_path, caption=draft.sender_logo_caption),
        ImageSlot(slot_id="logo_receiver", source_type="receiver_logo",
                  blob_path=receiver_logo_path, caption=draft.receiver_logo_caption),
        ImageSlot(slot_id="hero_contextual", source_type="contextual",
                  blob_path=None, caption=draft.contextual_image_caption),  # known gap — see markdown above
    ]

# Called for real in Section 7, once a draft exists to pull captions from.

## Section 4 — Prompt construction

One system prompt (the domain-bridging + grounding rules, fixed across every call) and one user prompt (this specific pair's sender + receiver context, plus optional feedback). Per `decision-log.md` §6: **single-shot, sequential generation** — one call producing the whole structured draft, not parallel per-section calls, because headline/body/CTA need to stay coherent (same pain point referenced throughout, consistent tone).

In [ ]:
SYSTEM_PROMPT = """You are a B2B marketing copywriter for a marketing agency. You write short, \
factually grounded outbound marketing articles that bridge a SENDER company (what they sell) and a \
RECEIVER company (who the article is being written for).

Rules:
1. Every factual claim must be grounded in the SENDER CONTEXT or RECEIVER CONTEXT provided in the \
user message. Do not invent facts, statistics, product features, or company details that are not \
present in that context.
2. The article must draw from BOTH sides — at least one body section should reference a specific \
receiver pain point, and at least one should reference a specific sender capability that addresses \
it. Do not write generic copy that could apply to any receiver.
3. Tone: persuasive, professional outbound marketing collateral — not a dry summary and not an RFP \
response.
4. Word limits (hard constraints, checked automatically after you respond — stay under these):
   - headline: <= 12 words
   - subheadline: <= 20 words (optional — return null if it doesn't add value)
   - each body_sections[].text: <= 150 words
   - body_sections: exactly 2 or 3 sections
   - call_to_action: <= 25 words
5. sender_logo_caption / receiver_logo_caption: just the company's display name.
6. contextual_image_caption: describe what a supporting hero image for this article should depict \
(no image is actually generated from this caption in this prototype — see the notebook's Section 3).
"""


def format_context_for_prompt(ctx: dict) -> str:
    """Turns one side's context bundle (Section 2's fetch_context() output) into a single plain-text
    block for a chat message. Inlines the parsed PDF text and, right after it, every extracted table
    rendered as simple "|"-separated rows — so facts that only exist in a table (e.g. Ferrow's
    pain-point/severity table) are visible to the model as readable text, not lost as structured
    data it never sees. Deliberately ignores ctx["image_paths"] — images are never sent to the LLM as
    text; they're resolved separately in Section 3, after the LLM call, from blob paths this code
    already has."""
    lines = [ctx["text"]]
    for i, table in enumerate(ctx["tables"], start=1):
        lines.append(f"\n[Table {i}]")
        for row in table:
            lines.append(" | ".join(row))
    return "\n".join(lines)


def build_user_prompt(sender_ctx: dict, receiver_ctx: dict, feedback: str | None = None) -> str:
    """Assembles the full user-turn message sent alongside SYSTEM_PROMPT: a labeled SENDER CONTEXT
    block, a labeled RECEIVER CONTEXT block (both via format_context_for_prompt above), an optional
    FEEDBACK block (a plain string from a prior /evaluate call — decision-log.md §9's feedback
    contract; /generate never calls /evaluate itself, it just accepts this if the caller supplies
    it), and a one-line closing instruction. The explicit "=== SENDER CONTEXT ===" / "=== RECEIVER
    CONTEXT ===" labels exist so the model never has to guess which company is doing the selling vs.
    which is being pitched to — this is the actual mechanism behind the domain-bridging requirement
    (decision-log.md: "an article must draw facts from both sender and receiver context, not just
    one"), not just the SYSTEM_PROMPT's rule #2 asking nicely."""
    parts = [
        "=== SENDER CONTEXT (the company being marketed — what they sell) ===\n"
        + format_context_for_prompt(sender_ctx),
        "=== RECEIVER CONTEXT (the pitch target — who this article is for) ===\n"
        + format_context_for_prompt(receiver_ctx),
    ]
    if feedback:
        parts.append(
            "=== FEEDBACK FROM A PRIOR REVIEW — address this specifically in the new draft ===\n"
            + feedback
        )
    parts.append("Write the article now. Ground every claim in the context above — do not invent facts.")
    return "\n\n".join(parts)

In [ ]:
preview_prompt = build_user_prompt(sender_ctx, receiver_ctx)
print(f"System prompt: {word_count(SYSTEM_PROMPT)} words\n")
print(SYSTEM_PROMPT)
print("\n" + "=" * 80 + "\n")
print(f"User prompt: {word_count(preview_prompt)} words\n")
print(preview_prompt)

## Pretext — traditional prompt-only JSON vs. structured outputs, side by side

Two small, real calls to `gpt-5-mini` — same task, same model, two different mechanisms for getting JSON back — so the difference from the last two questions is visible instead of just described. `TinyDraft` below is a deliberately tiny stand-in for `LLMDraft` (3 fields, one validator, reusing Section 1's `word_count()` so this stays short) — not a real contract, just enough to see the mechanism.

**Note on `response_format` vs `text_format`**: OpenAI has two API surfaces — the older **Chat Completions API** (`client.chat.completions.create/parse`, parameter `response_format`) and the newer **Responses API** (`client.responses.create/parse`, parameter `text_format`). Both support structured outputs the same way underneath; they're just two different client methods with different parameter names. This notebook (both cells below, and Section 5) uses Chat Completions, matching what's already built. Microsoft's current docs recommend the Responses API for new projects — worth a separate conversation on whether to migrate before this gets extracted into `app/generation.py`, not conflated into this comparison.

Section 5 right after this does exactly what Cell B below does, for real, with the full `LLMDraft` schema instead of `TinyDraft`.

In [ ]:
from openai import AzureOpenAI
import json as _json

# Inline client here — Section 5 below formally defines get_client(), but that cell hasn't run yet
# at this point in the notebook, so this cell builds its own for now.
_client = AzureOpenAI(
    azure_endpoint=settings.azure_openai_endpoint,
    api_key=settings.azure_openai_api_key,
    api_version=settings.azure_openai_api_version,
)


class TinyDraft(BaseModel):
    """Deliberately tiny stand-in for LLMDraft — just enough fields to see the mechanism, not a
    real contract. Reused by Cell B below, unchanged."""
    headline: str
    subheadline: str
    call_to_action: str

    @field_validator("headline")
    @classmethod
    def _headline_max_12(cls, v: str) -> str:
        if word_count(v) > 12:  # reusing Section 1's helper — no new validation logic here
            raise ValueError(f"headline must be <=12 words, got {word_count(v)}")
        return v


# --- Traditional way: describe the shape in the prompt, ask nicely, parse + validate it ourselves ---
traditional_system_prompt = (
    "You are a marketing copywriter. Respond with ONLY a JSON object, no other text, matching "
    'exactly this shape: {"headline": "<string, at most 12 words>", "subheadline": "<string>", '
    '"call_to_action": "<string>"}'
)

completion = _client.chat.completions.create(  # .create(), not .parse() — no response_format at all
    model=settings.azure_openai_deployment_name,
    messages=[
        {"role": "system", "content": traditional_system_prompt},
        {"role": "user", "content": "Write a short pitch bridging Northbridge Analytics and Ferrow Industrial Group."},
    ],
)
raw_text = completion.choices[0].message.content
print("Raw text the model returned:\n", raw_text)

# We have to parse and validate this ourselves — exactly the step response_format eliminates in Cell B.
try:
    traditional_draft = TinyDraft(**_json.loads(raw_text))
    print("\nParsed + validated OK:", traditional_draft)
except Exception as exc:
    print("\nFAILED to parse/validate:", type(exc).__name__, "-", exc)

In [ ]:
# --- Structured outputs way: schema is a separate, enforced parameter — nothing "asked for" in the prompt ---
completion = _client.beta.chat.completions.parse(  # .parse(), with response_format
    model=settings.azure_openai_deployment_name,
    messages=[
        {"role": "system", "content": "You are a marketing copywriter."},  # no JSON-shape instructions needed
        {"role": "user", "content": "Write a short pitch bridging Northbridge Analytics and Ferrow Industrial Group."},
    ],
    response_format=TinyDraft,  # reuses the exact same class Cell A defined — no redefinition
)
structured_draft = completion.choices[0].message.parsed  # already a validated TinyDraft object — no manual json.loads/try-except
print(structured_draft)

## Section 5 — Azure OpenAI structured-output call

This is the real call. `client.beta.chat.completions.parse(..., response_format=LLMDraft)` sends `LLMDraft`'s JSON Schema (printed in Section 1) to Azure OpenAI, which is contractually guaranteed (structured-output / strict mode) to return JSON matching that shape — right field names, right types, right array bounds. It is **not** guaranteed to respect the word-count rules from the system prompt, since those aren't expressible in JSON Schema — that gap is exactly what Section 6's retry loop exists for.

**Gotcha**: the `model=` parameter is the Azure **deployment name** (`gpt-5-mini`, set in Section 0 step 2), not the underlying model family name — Azure OpenAI routes by deployment, unlike OpenAI's own API which routes by model name directly.

In [ ]:
from openai import AzureOpenAI

def get_client() -> AzureOpenAI:
    """Builds the Azure OpenAI SDK client from Settings. The assert below exists so a missing
    endpoint/key fails immediately with an actionable message ("finish Section 0, fill .env, restart
    the kernel") instead of surfacing as a confusing 401/connection error several lines deeper inside
    the SDK. Not cached — built fresh on every call, same deliberate simplicity trade-off as
    Section 2's _table_client()."""
    assert settings.azure_openai_endpoint and settings.azure_openai_api_key, (
        "AZURE_OPENAI_ENDPOINT / AZURE_OPENAI_API_KEY not set. Finish Section 0's provisioning "
        "steps, fill .env, then restart this kernel."
    )
    return AzureOpenAI(
        azure_endpoint=settings.azure_openai_endpoint,
        api_key=settings.azure_openai_api_key,
        api_version=settings.azure_openai_api_version,
    )


def generate_draft(sender_ctx: dict, receiver_ctx: dict, feedback: str | None = None) -> LLMDraft:
    """The one Azure OpenAI call in this whole notebook — the only function here that spends real
    API quota. Builds the user prompt (above), sends it with the fixed SYSTEM_PROMPT to the deployed
    model via structured-output mode (response_format=LLMDraft — see Section 5's markdown for what
    that actually guarantees vs. doesn't), checks for a content-policy refusal before trusting the
    result, and returns the parsed + validated LLMDraft. Deliberately has no retry logic of its own —
    that's Section 6's generate_draft_with_retry() wrapper; this function's only job is one prompt
    in, one validated draft out."""
    client = get_client()
    user_prompt = build_user_prompt(sender_ctx, receiver_ctx, feedback)
    completion = client.beta.chat.completions.parse(
        model=settings.azure_openai_deployment_name,  # Azure: deployment name, not base model name
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt},
        ],
        response_format=LLMDraft, #THIS IS WHERE ENFORCEMENT HAPPENS — the model is never shown article_id, version, sender_id/receiver_id, or theme, so it cannot hallucinate or overwrite them
    )
    choice = completion.choices[0]
    if choice.message.refusal:
        raise RuntimeError(f"Model refused: {choice.message.refusal}")
    return choice.message.parsed

In [ ]:
draft = generate_draft(sender_ctx, receiver_ctx)
print(draft.model_dump_json(indent=2))
print(f"\nheadline: {word_count(draft.headline)} words")
if draft.subheadline:
    print(f"subheadline: {word_count(draft.subheadline)} words")
for i, section in enumerate(draft.body_sections, start=1):
    print(f"body_sections[{i}] '{section.heading}': {word_count(section.text)} words")
print(f"call_to_action: {word_count(draft.call_to_action)} words")

## Section 6 — Validation and repair: not blind trust in the LLM's JSON

Per `decision-log.md` §6: on a validation failure (a word-limit `field_validator` raising, from Section 1), the `ValidationError` message is fed back to the model as extra instruction and it gets **one retry** (2 attempts total) before this gives up loudly. This is a real self-healing pattern, not a hand-wave — demonstrated below without spending an API call, by handing `LLMDraft` a deliberately too-long headline directly.

In [ ]:
MAX_ATTEMPTS = 2


def generate_draft_with_retry(sender_ctx: dict, receiver_ctx: dict, feedback: str | None = None) -> tuple[LLMDraft, int]:
    """Returns (validated draft, attempts_used)."""
    last_error = None
    current_feedback = feedback
    for attempt in range(1, MAX_ATTEMPTS + 1):
        try:
            return generate_draft(sender_ctx, receiver_ctx, feedback=current_feedback), attempt
        except ValidationError as exc:
            last_error = exc
            print(f"Attempt {attempt} failed word-limit/schema validation:\n{exc}\n")
            retry_note = f"Your previous draft failed validation — fix this and resubmit:\n{exc}"
            current_feedback = f"{feedback}\n\n{retry_note}" if feedback else retry_note
    raise RuntimeError(f"Gave up after {MAX_ATTEMPTS} attempts. Last error:\n{last_error}")

In [ ]:
# Dry run of the failure path — no API call. This is exactly the kind of ValidationError
# generate_draft_with_retry above catches and turns into retry feedback for the model.
bad_draft = {
    "headline": "This Headline Is Deliberately Written With Far Too Many Words To Pass Validation",
    "subheadline": None,
    "body_sections": [
        {"heading": "Section One", "text": "Short body text."},
        {"heading": "Section Two", "text": "Also short."},
    ],
    "call_to_action": "Act now.",
    "sender_logo_caption": "Northbridge Analytics",
    "receiver_logo_caption": "Ferrow Industrial Group",
    "contextual_image_caption": "A dashboard over a plant floor.",
}

try:
    LLMDraft(**bad_draft)
except ValidationError as exc:
    print("Caught exactly the error generate_draft_with_retry would feed back to the model:\n")
    print(exc)

## Section 7 — Assembling the full `ArticleOutput`

Everything SYSTEM-ASSIGNED gets filled in here — `article_id`, `version` (auto-incremented per pair, looked up from a new `articles` Table Storage table, sibling to `documentmetadata`), `total_word_count`, `created_at`. `total_word_count`'s 300–600 target is a **soft** check (the schema PDF says "target", not a hard per-field limit like the others) — flagged with a warning, not raised as an error, since the per-section hard limits already bound it loosely (2–3 sections × ≤150 words + headline/subheadline/CTA caps → roughly 40–490 words structurally, so 300–600 is a quality target to watch, not a contract violation to retry over).

In [ ]:
# A second Azure Table Storage table — NOT a Blob container, NOT a folder, NOT a filesystem path.
# Same account (the storage account) and same PartitionKey/RowKey model as "documentmetadata" (Section 2), just
# a different table, for a different kind of row (generated articles instead of uploaded documents).
# Created automatically the first time _table_client(ARTICLES_TABLE) runs, same idempotent
# create_table_if_not_exists() pattern as every other table/blob container in this project — nothing
# to create by hand. Once at least one article has been saved, find it in the Portal at: Storage
# Account "the storage account" -> Storage browser -> Tables -> "articles" (a sibling to "documentmetadata",
# not inside it).
ARTICLES_TABLE = "articles"

def get_next_version(sender_id: str, receiver_id: str) -> int:
    """Computes ArticleOutput.version (SYSTEM-ASSIGNED) — implements the schema PDF's "auto-
    incremented per sender/receiver pair" rule, so regenerating for the same pair (e.g. after a
    future /evaluate round trip) produces v2, v3, ... instead of silently overwriting v1.

    Mechanically: query the "articles" table (Table Storage, not Blob Storage — there is no file
    path involved anywhere here) for every row that shares this pair's PartitionKey
    (f"{sender_id}__{receiver_id}", same convention as Section 2's fetch_context) — i.e. every
    article ever generated for this exact pair, regardless of version. "existing" is that list of
    rows. If it's empty ("not existing" — no article has ever been generated for this pair before),
    this is the first one, so version 1. Otherwise, take the highest version number already saved
    and add 1 — so this is always a READ (a lookup of past articles), performed *before* the new
    article is created, purely to decide what number to stamp on it. It does not save anything
    itself; save_article() below is the separate function that writes.
    """
    partition_key = f"{sender_id}__{receiver_id}"
    table = _table_client(ARTICLES_TABLE)
    existing = list(table.query_entities(query_filter=f"PartitionKey eq '{partition_key}'"))
    if not existing:
        return 1
    return max(e["version"] for e in existing) + 1


def save_article(article: ArticleOutput) -> None:
    """Persists one already-fully-built ArticleOutput as a single row in the "articles" table — a
    WRITE, not a lookup (get_next_version above is the read; this is the separate write). Same
    account as "documentmetadata", and the same justification from decision-log.md §16 for Table
    Storage over Cosmos DB/Postgres applies here too: the only way this table is ever read is an
    exact-ID lookup (a specific sender_id/receiver_id/article_id, e.g. by a future /evaluate call
    fetching one specific article to review) — never a full-text or semantic search — so Table
    Storage's simple key-value model is a direct fit, and a heavier database would be paying for
    query capabilities nothing here uses.

    `entity` is a plain dict because that's what the azure-data-tables SDK's upsert_entity() expects
    — PartitionKey/RowKey are the required lookup keys (same pairing scheme as everywhere else:
    PartitionKey group by sender/receiver pair, RowKey = this specific article's own UUID).
    `article_json` stores the ENTIRE ArticleOutput as one serialized JSON string (via
    model_dump_json()) in a single column — simplest possible way to keep the full response
    retrievable later without modeling every nested field as its own Table column. `created_at` is
    converted to `.isoformat()` (a plain string like "2026-07-26T14:32:00+00:00") rather than left as
    a Python datetime object, so it's a plain, unambiguous, human-readable value if anyone opens this
    row directly in the Storage Browser.

    upsert (not a strict insert) is used defensively: RowKey is a freshly-generated UUID
    (article.article_id) every time, so in practice this always creates a new row — but upsert means
    an accidental duplicate call (e.g. a retried HTTP request) overwrites harmlessly instead of
    erroring.
    """
    entity = {
        "PartitionKey": f"{article.sender_id}__{article.receiver_id}",
        "RowKey": str(article.article_id),
        "version": article.version,
        "article_json": article.model_dump_json(),
        "created_at": article.created_at.isoformat(),
    }
    _table_client(ARTICLES_TABLE).upsert_entity(entity)


def assemble_article(request: GenerateRequest, draft: LLMDraft, image_slots: list[ImageSlot]) -> ArticleOutput:
    """The "backend fills in everything else" step from Section 1's workflow diagram — this is the
    direct answer to "where did Section 6 leave off, and what's next": Section 6 ends with nothing
    more than a validated `draft` (an LLMDraft — headline/subheadline/body_sections/call_to_action/
    captions only, per Section 1). That alone is not a usable API response — it has no article_id,
    no version, no theme, no image_slots, no total_word_count, no created_at. This function is what
    turns "a validated LLM draft" into "the actual, complete ArticleOutput /generate returns," by
    combining three different sources into one object:
      1. draft's fields (LLM-GENERATED) — copied straight across, unchanged.
      2. request's fields (INPUT PARAMETER: sender_id, receiver_id, theme) — echoed straight through
         from whatever the caller originally sent.
      3. Freshly computed, backend-only values (SYSTEM-ASSIGNED) — article_id (a new uuid4()),
         version (get_next_version(), above), total_word_count (computed here), created_at (now).
    image_slots (built separately in Section 3, since it needs draft's captions but doesn't need
    anything else assembled here) is passed in already-built.
    """
    total_words = (
        word_count(draft.headline)
        + (word_count(draft.subheadline) if draft.subheadline else 0)
        + sum(word_count(s.text) for s in draft.body_sections)
        + word_count(draft.call_to_action)
    )
    if not (300 <= total_words <= 600):
        print(f"Note: total_word_count={total_words} is outside the 300-600 soft target.")

    return ArticleOutput(
        article_id=uuid4(),
        sender_id=request.sender_id,
        receiver_id=request.receiver_id,
        version=get_next_version(request.sender_id, request.receiver_id),
        headline=draft.headline,
        subheadline=draft.subheadline,
        body_sections=draft.body_sections,
        call_to_action=draft.call_to_action,
        image_slots=image_slots,
        theme=request.theme,
        total_word_count=total_words,
        created_at=datetime.now(timezone.utc),
    )

In [ ]:
request = GenerateRequest(
    sender_id=SENDER_ID,
    receiver_id=RECEIVER_ID,
    # Same theme values as the worked example in docs/architecture/api-payload-schemas.drawio, so this run's
    # output can be compared directly against that already-built reference rendering.
    theme=ThemeColors(primary_color="#1B3A5C", secondary_color="#2FA8A0", accent_color="#F2A93B"),
)

image_slots = resolve_image_slots(sender_ctx, receiver_ctx, draft)
article = assemble_article(request, draft, image_slots)

print(article.model_dump_json(indent=2))

## Section 8 — Persist

In [ ]:
save_article(article)
print(f"Saved article_id={article.article_id} version={article.version} "
      f"to Table Storage ('{ARTICLES_TABLE}' table, PartitionKey={article.sender_id}__{article.receiver_id})")

## Section 9 — Now the real end-to-end function

Same pattern as `upload_pipeline_walkthrough.ipynb`'s Step 4: everything above was done by hand to show each piece. This function is what `POST /generate` will actually call once extracted into `app/routers/generate.py` — confirm it reproduces the same result end to end.

In [ ]:
def generate_article(request: GenerateRequest) -> ArticleOutput:
    sender_ctx = fetch_context(request.sender_id, request.receiver_id, role="sender")
    receiver_ctx = fetch_context(request.sender_id, request.receiver_id, role="receiver")
    draft, attempts = generate_draft_with_retry(sender_ctx, receiver_ctx, feedback=request.feedback)
    image_slots = resolve_image_slots(sender_ctx, receiver_ctx, draft)
    article = assemble_article(request, draft, image_slots)
    save_article(article)
    print(f"(used {attempts}/{MAX_ATTEMPTS} generation attempt(s))")
    return article

In [ ]:
# Re-running for the same pair — watch `version` increment automatically.
article_v2 = generate_article(request)
print(f"\narticle_id={article_v2.article_id}  version={article_v2.version}  "
      f"total_word_count={article_v2.total_word_count}")
print(f"\nheadline: {article_v2.headline}")

## Section 10 — Reusability: same sender, different receiver

If you've also uploaded context for a second receiver (e.g. `cascade-logistics`, per `decision-log.md` §14's "1 sender × 2 receivers" mock data), this demonstrates the sender context being reused as-is while the receiver side — and therefore the generated bridge — changes completely. If you haven't uploaded that pair yet, `fetch_context`'s error message above tells you exactly what to run first.

In [ ]:
second_request = GenerateRequest(
    sender_id="northbridge-analytics",
    receiver_id="cascade-logistics",
    theme=ThemeColors(primary_color="#14508C", secondary_color="#2E9E5B", accent_color="#F2A93B"),
)

second_article = generate_article(second_request)
print(second_article.model_dump_json(indent=2))

## Wrap-up

What ran, end to end, against real Azure resources:

1. **Table Storage read** — `fetch_context()` pulled the already-uploaded sender + receiver documents by `(sender_id, receiver_id, role)`.
2. **Azure OpenAI structured-output call** — `generate_draft()` produced a schema-guaranteed `LLMDraft`.
3. **Python-side validation** — word-limit `field_validator`s on `LLMDraft`, with `generate_draft_with_retry()` as the repair loop for anything the JSON Schema itself couldn't constrain.
4. **Backend assembly** — `assemble_article()` filled in every SYSTEM-ASSIGNED field; the LLM never saw or touched `article_id`, `version`, or `theme`.
5. **Table Storage write** — `save_article()` persisted the result to a new `articles` table.

**Known gaps surfaced along the way** (name these proactively in the presentation rather than waiting for Q&A to expose them):
- No contextual/hero image asset pipeline — `hero_contextual.blob_path` is always `None` (Section 3).
- No image captioning/vision pass on ingestion (already flagged in `decision-log.md` §7).
- `total_word_count`'s 300-600 target is a soft warning, not enforced.

**Next steps, in order:**
1. You run this notebook, tune the prompt wording / retry behavior / anything that reads oddly.
2. Once it's producing good output consistently, say so — this logic gets extracted into `app/models.py` (the Pydantic models above), `app/generation.py` (context fetch, prompt, Azure OpenAI call, retry), and `app/routers/generate.py` (the thin FastAPI wrapper, same pattern as `app/routers/upload.py`), then wired into `app/main.py`.
3. `CODE_MAP.md` gets a new `## POST /generate` section, same format as the existing `## POST /upload` one.
4. `azure-setup-log.md` §4 gets updated once the Azure OpenAI resource is actually provisioned.